In [2]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import T5ForConditionalGeneration, T5Tokenizer
from sklearn.model_selection import train_test_split
from rouge_score import rouge_scorer
import numpy as np

### Load Data

In this version, we filtered the data to eliminate things the model struggled with:
1. Code heavy answers
2. Long complex prompts

In [3]:
df = pd.read_csv("prompt_answer_pairs_clean.csv")
print(f"Original: {len(df)} rows")

# 1. Filter out rows where answer is mostly code blocks
df = df[~df["answer"].str.contains(r'(\[CODE_BLOCK_\d+\].*){3,}', regex=True)]
print(f"After code block filter: {len(df)} rows")

# 2. Filter out very long prompts (over 50 words)
df = df[df["prompt"].str.split().str.len() <= 50]
print(f"After long prompt filter: {len(df)} rows")

# 3. Filter out very short prompts (less than 3 words)
df = df[df["prompt"].str.split().str.len() >= 3]
print(f"After short prompt filter: {len(df)} rows")

# 4. Filter out very long answers (over 400 words)
df = df[df["answer"].str.split().str.len() <= 400]
print(f"After long answer filter: {len(df)} rows")

# 5. Drop any rows where answer still has only code blocks and nothing else
df = df[~df["answer"].str.fullmatch(r'(\[CODE_BLOCK_\d+\]\s*)+', na=False)]
print(f"After code-only answer filter: {len(df)} rows")

df.to_csv("prompt_answer_pairs_filtered.csv", index=False, encoding="utf-8")

Original: 8058 rows


/tmp/ipykernel_18636/1155999669.py:7: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df = df[~df["answer"].str.contains(r'(\[CODE_BLOCK_\d+\].*){3,}', regex=True)]


After code block filter: 7510 rows
After long prompt filter: 4389 rows
After short prompt filter: 4357 rows
After long answer filter: 4056 rows
After code-only answer filter: 3993 rows


In [4]:
df = pd.read_csv("prompt_answer_pairs_filtered.csv")[["prompt", "answer"]].dropna()

# Convert to strings
df["prompt"] = df["prompt"].astype(str).str.strip()
df["answer"] = df["answer"].astype(str).str.strip()

# 70/15/15
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42)
val_df,   test_df = train_test_split(temp_df, test_size=0.50, random_state=42)

### Tokenize data

In [5]:
def tokenize_data(df):
  # Input is the answer, tokenizes answer text
  inputs = tokenizer(
      ["predict prompt: " + text for text in df["answer"].tolist()],
      max_length=512,
      padding="max_length",
      truncation=True,
      return_tensors="pt"
    )
  # Target is the prompt, tokenizes prompt text
  targets = tokenizer(
      df["prompt"].tolist(),
      max_length=512,
      padding="max_length",
      truncation=True,
      return_tensors="pt"
  )

  labels = targets["input_ids"]
  # Replace padding tokens with -100 so model does not try to predict them
  labels[labels == tokenizer.pad_token_id] = -100

  # Create tokenized dataset
  dataset = torch.utils.data.TensorDataset(
      inputs["input_ids"],
      inputs["attention_mask"],
      labels
  )
  return dataset

In [6]:
# Load tokenizer
tokenizer = T5Tokenizer.from_pretrained("t5-small")
# Model is a T5 language model
# T5 is a seq2seq model that is pretrained using a  using a denoising objective
model     = T5ForConditionalGeneration.from_pretrained("t5-small").to(torch.device("cuda"))

# Tokenize all datasets
train_dataset = tokenize_data(train_df)
val_dataset   = tokenize_data(val_df)
test_dataset  = tokenize_data(test_df)

# DataLoader does batching, shuffling and makes dataset iterable
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=8)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

### Evaluate how model is doing on validation set

In [7]:
def evaluate(loader):
  # Switch model to evaluation mode
  model.eval()
  total_loss = 0
  # Do not track gradients to save memory
  with torch.no_grad():
    # For every batch accumulate the loss
    for batch in loader:
        input_ids, attention_mask, labels = batch
        input_ids      = input_ids.to(torch.device("cuda"))
        attention_mask = attention_mask.to(torch.device("cuda"))
        labels         = labels.to(torch.device("cuda"))
        loss = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels).loss
        total_loss += loss.item()
  # Return the average loss per epoch
  return total_loss / len(loader)

### Train Model

In [8]:
# Load optimizer
optimizer   = torch.optim.AdamW(model.parameters(), lr=5e-5)
best_val    = float("inf")

for epoch in range(5):
  # Switch model to train mode
  model.train()
  total_train_loss = 0

  for i, batch in enumerate(train_loader):
    input_ids, attention_mask, labels = batch
    input_ids      = input_ids.to(torch.device("cuda"))
    attention_mask = attention_mask.to(torch.device("cuda"))
    labels         = labels.to(torch.device("cuda"))

    # Clear gradient from previous batch
    optimizer.zero_grad()
    # Calculates loss using T5 model
    loss = model(input_ids=input_ids,
                  attention_mask=attention_mask,
                  labels=labels).loss
    # Backpropagation
    loss.backward()
    # Gradient clipping (caps gradients at 1)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    # Updates model weights using gradients calculated by loss.backward
    optimizer.step()

    total_train_loss += loss.item()

    if (i + 1) % 50 == 0:
      print(f"  Epoch {epoch+1} | Step {i+1}/{len(train_loader)} | Loss: {loss.item():.4f}")

  avg_train = total_train_loss / len(train_loader)
  avg_val   = evaluate(val_loader)
  print(f"\nEpoch {epoch+1}/5 | Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f}")

  # Gets best model by updating model saved when average validation loss is less
  if avg_val < best_val:
    best_val = avg_val
    model.save_pretrained("prompt_predictor")
    tokenizer.save_pretrained("prompt_predictor")

  Epoch 1 | Step 50/350 | Loss: 3.6245
  Epoch 1 | Step 100/350 | Loss: 3.3072
  Epoch 1 | Step 150/350 | Loss: 3.2638
  Epoch 1 | Step 200/350 | Loss: 4.1019
  Epoch 1 | Step 250/350 | Loss: 3.0570
  Epoch 1 | Step 300/350 | Loss: 3.0996
  Epoch 1 | Step 350/350 | Loss: 4.3806

Epoch 1/5 | Train Loss: 3.4554 | Val Loss: 2.9361


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Epoch 2 | Step 50/350 | Loss: 3.3625
  Epoch 2 | Step 100/350 | Loss: 3.5716
  Epoch 2 | Step 150/350 | Loss: 2.8952
  Epoch 2 | Step 200/350 | Loss: 2.8343
  Epoch 2 | Step 250/350 | Loss: 2.4661
  Epoch 2 | Step 300/350 | Loss: 3.0490
  Epoch 2 | Step 350/350 | Loss: 4.1110

Epoch 2/5 | Train Loss: 3.1849 | Val Loss: 2.8648


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Epoch 3 | Step 50/350 | Loss: 3.4625
  Epoch 3 | Step 100/350 | Loss: 3.2883
  Epoch 3 | Step 150/350 | Loss: 3.5114
  Epoch 3 | Step 200/350 | Loss: 3.5952
  Epoch 3 | Step 250/350 | Loss: 4.0213
  Epoch 3 | Step 300/350 | Loss: 2.7665
  Epoch 3 | Step 350/350 | Loss: 2.9231

Epoch 3/5 | Train Loss: 3.0735 | Val Loss: 2.8265


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Epoch 4 | Step 50/350 | Loss: 3.9540
  Epoch 4 | Step 100/350 | Loss: 2.6511
  Epoch 4 | Step 150/350 | Loss: 2.8787
  Epoch 4 | Step 200/350 | Loss: 2.1830
  Epoch 4 | Step 250/350 | Loss: 2.7582
  Epoch 4 | Step 300/350 | Loss: 3.4823
  Epoch 4 | Step 350/350 | Loss: 1.5360

Epoch 4/5 | Train Loss: 3.0031 | Val Loss: 2.7990


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Epoch 5 | Step 50/350 | Loss: 3.1127
  Epoch 5 | Step 100/350 | Loss: 3.2132
  Epoch 5 | Step 150/350 | Loss: 3.4052
  Epoch 5 | Step 200/350 | Loss: 2.8642
  Epoch 5 | Step 250/350 | Loss: 2.6858
  Epoch 5 | Step 300/350 | Loss: 3.1490
  Epoch 5 | Step 350/350 | Loss: 2.8881

Epoch 5/5 | Train Loss: 2.9196 | Val Loss: 2.7785


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

### Evaulate Model Performace with ROUGE
ROUGE is a metric that measures how simular predicted prompt is to real prompt by comparing n-gram overlap

- ROUGE-1: Counts overlap of individual words
- ROUGE-2: Counts overlap of word pairs
- ROUGE-L: Finds longest common sequence of words in order

In [9]:
model = T5ForConditionalGeneration.from_pretrained("prompt_predictor").to(torch.device("cuda"))
model.eval()

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
r1, r2, rL = [], [], []

for _, row in test_df.iterrows():
    # Tokenize input
    input_text = "predict prompt: " + row["answer"]
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(torch.device("cuda"))

    # Generate prediction
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_length=128,
            num_beams=4,
            early_stopping=True
        )

    # Decode prediction back to text
    predicted = tokenizer.decode(output[0], skip_special_tokens=True)

    # Score predicted vs real prompt
    scores = scorer.score(row["prompt"], predicted)
    r1.append(scores["rouge1"].fmeasure)
    r2.append(scores["rouge2"].fmeasure)
    rL.append(scores["rougeL"].fmeasure)

# Print results
print(f"ROUGE-1:  {np.mean(r1):.4f}")
print(f"ROUGE-2:  {np.mean(r2):.4f}")
print(f"ROUGE-L:  {np.mean(rL):.4f}")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

ROUGE-1:  0.3369
ROUGE-2:  0.1690
ROUGE-L:  0.2979


### Sample Predictions

In [11]:
for _, row in test_df.head(20).iterrows():
    input_text = "predict prompt: " + row["answer"]
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(torch.device("cuda"))

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_length=128,
            num_beams=4,
            early_stopping=True
        )

    predicted = tokenizer.decode(output[0], skip_special_tokens=True)

    print(f"\nAnswer (truncated): {row['answer'][:200]}")
    print(f"Real prompt:        {row['prompt']}")
    print(f"Predicted prompt:   {predicted}")
    print(f"ROUGE-L:            {scorer.score(row['prompt'], predicted)['rougeL'].fmeasure:.4f}")


Answer (truncated): Absolutely, I'd be glad to help you start with automated testing. In Python, a commonly used library for creating automated tests is unittest. This library allows you to create test cases, assert the 
Real prompt:        Thanks again!  Now, can you please help me develop an automated test process for these rules?  I have not created automated tests before and I need help, please.
Predicted prompt:   How do I start with automated testing?
ROUGE-L:            0.1714

Answer (truncated): Ozone dissolves in water and can be used for certain applications, such as water purification. However, it's important to note that ozone concentrations in water can still be harmful if not properly r
Real prompt:        What if Input it in water?
Predicted prompt:   What is ozone in water?
ROUGE-L:            0.5455

Answer (truncated): There are several open-source and plain text file formats that can be used for creating presentations. These include:.odp (OpenDocument Presentation)